In [5]:
# ============================================================
# PopQA EM Evaluation — OpenAI Version (Clean & Stable)
# ============================================================

from __future__ import annotations
import os
import time
from contextlib import nullcontext

from dotenv import load_dotenv
from openai import OpenAI
from tqdm.auto import tqdm
from datasets import load_dataset

from fuseqa import *

import logging
logging.getLogger("httpx").setLevel(logging.WARNING)

# ------------------------------------------------------------
# Load environment
# ------------------------------------------------------------
load_dotenv()

client = OpenAI()
MODEL_NAME = "gpt-3.5-turbo"  # or "gpt-4o-mini"

print(f"Using model: {MODEL_NAME}")




Using model: gpt-3.5-turbo


In [20]:
# ------------------------------------------------------------
# Dataset
# ------------------------------------------------------------
ds = load_dataset("MinaGabriel/popqa-retrieval20-with-sre")["test"] 
START_IDX = 10951  # resume point

for i in range(START_IDX, len(ds)):
    item = ds[i]
    question = item["question"]  
    print(f"\n=== Question {question} ===")



=== Question Who was the composer of The Moment's Energy? ===

=== Question Who was the composer of Touch? ===

=== Question Who was the composer of Aladdin – Original Motion Picture Soundtrack? ===

=== Question Who was the composer of Das neugeborne Kindelein, BWV 122? ===

=== Question Who was the composer of Prince? ===

=== Question Who was the composer of Fire? ===

=== Question Who was the composer of Going Up? ===

=== Question Who was the composer of Violin Concerto No. 1? ===

=== Question Who was the composer of Serial? ===

=== Question Who was the composer of Gang? ===

=== Question Who was the composer of Locus iste? ===

=== Question Who was the composer of Beautiful? ===

=== Question Who was the composer of Apollo? ===

=== Question Who was the composer of Listen? ===

=== Question Who was the composer of So What? ===

=== Question Who was the composer of Memories? ===

=== Question Who was the composer of Raising the Wind? ===

=== Question Who was the composer of Pe

In [7]:
def prompt_fn(question: str, context: str, use_context: bool):
    if use_context:
        return (
            f"Context: {context}\n\n"
            f"Q: {question}\n"
            "A (one word from context):"
        )
    else:
        return (
            f"Q: {question}\n"
            "A (one word only):"
        )

In [9]:

# ------------------------------------------------------------
# OpenAI LLM Call
# ------------------------------------------------------------
def ask_llm_openai(question, context, use_context, print_prompt=False):
    prompt = prompt_fn(question, context, use_context)

    if print_prompt:
        print("=== PROMPT ===")
        print(prompt)

    resp = client.responses.create(
        model=MODEL_NAME,
        input=prompt,
        max_output_tokens=32,
        #reasoning={"effort": "minimal"}
    )

    pred = (resp.output_text or "").strip().rstrip(".")

    if print_prompt:
        print("=== PREDICTION ===")
        print(pred)

    return pred


# ------------------------------------------------------------
# Config
# ------------------------------------------------------------
TOP_K = 3
CLIP_CHARS = 300
SRE_SCORE_TH = 0.90

RUN_TYPE = "FUSEQA-SRE"  # "FUSEQA", "FUSEQA-SRE", "NO-CONTEXT"
USE_CONTEXT = RUN_TYPE in ("FUSEQA", "FUSEQA-SRE")

RESULTS_DIR = "results"
WRITE_OUTPUTS = True
PRINT_PROMPTS = False

os.makedirs(RESULTS_DIR, exist_ok=True)

file_name = hf_model_to_filename(MODEL_NAME + "-" + RUN_TYPE)
outfile = os.path.join(RESULTS_DIR, file_name + ".jsonl")

# ------------------------------------------------------------
# Metrics
# ------------------------------------------------------------
counts  = {g: 0 for g in ("ALL", "LONG-TAIL", "INFREQUENT", "FREQUENT")}
em_hits = {g: 0 for g in ("ALL", "LONG-TAIL", "INFREQUENT", "FREQUENT")}

def update_metrics(tier, em):
    counts["ALL"] += 1
    em_hits["ALL"] += em
    if tier in counts:
        counts[tier] += 1
        em_hits[tier] += em

def current_scores():
    return {
        "ALL_EM":     safe_div(em_hits["ALL"],        counts["ALL"]),
        "Long_Tail":  safe_div(em_hits["LONG-TAIL"],  counts["LONG-TAIL"]),
        "Infrequent": safe_div(em_hits["INFREQUENT"], counts["INFREQUENT"]),
        "Frequent":   safe_div(em_hits["FREQUENT"],   counts["FREQUENT"]),
    }

# ------------------------------------------------------------
# Run
# ------------------------------------------------------------
start_time = time.time()
tqdm._instances.clear()

with (open(outfile, "w", encoding="utf-8", buffering=1) if WRITE_OUTPUTS else nullcontext()) as writer:
    with tqdm(
        total=len(ds),
        desc="Generating + Evaluating (OpenAI)",
        dynamic_ncols=True,
        leave=False,
    ) as pbar:

        for i in range(len(ds)):
            ex = ds[i]

            q = ex["question"]
            s_pop = int(ex.get("s_pop", 0))
            tier = tier_from_spop(s_pop)

            gold = parse_list(ex.get("possible_answers"))
            gold_norm_set = {norm(g) for g in gold if norm(g)}

            context = get_context_for_run(
                ex, RUN_TYPE, USE_CONTEXT,
                top_k=TOP_K,
                sre_score_th=SRE_SCORE_TH,
                clip_chars=CLIP_CHARS
            )

            pred = ask_llm_openai(
                question=q,
                context=context,
                use_context=USE_CONTEXT,
                print_prompt=PRINT_PROMPTS,
            )

            pred_norm = norm(pred)
            em = int(pred_norm in gold_norm_set) if gold_norm_set else 0

            update_metrics(tier, em)

            if WRITE_OUTPUTS:
                write_record(writer, {
                    "i": i,
                    "s_pop": s_pop,
                    "tier": tier,
                    "question": q,
                    "gold": gold,
                    "pred": pred,
                    "em": em,
                })

            pbar.update(1)

            if i % 10 == 0:
                pbar.set_postfix({k: f"{v:.4f}" for k, v in current_scores().items()})

total_time = time.time() - start_time

generate_report(
    counts,
    em_hits,
    file_name,
    model_name=MODEL_NAME,
    run_type=RUN_TYPE,
    total_time=total_time,
)

Generating + Evaluating (OpenAI):   0%|          | 0/14267 [00:00<?, ?it/s]

INFO:openai._base_client:Retrying request to /responses in 0.435168 seconds
INFO:openai._base_client:Retrying request to /responses in 0.926747 seconds
INFO:openai._base_client:Retrying request to /responses in 0.382171 seconds
INFO:openai._base_client:Retrying request to /responses in 0.386972 seconds
INFO:openai._base_client:Retrying request to /responses in 0.418948 seconds
INFO:openai._base_client:Retrying request to /responses in 0.419611 seconds
INFO:openai._base_client:Retrying request to /responses in 0.763386 seconds
INFO:openai._base_client:Retrying request to /responses in 0.499229 seconds
INFO:openai._base_client:Retrying request to /responses in 0.964231 seconds
INFO:openai._base_client:Retrying request to /responses in 0.458270 seconds
INFO:openai._base_client:Retrying request to /responses in 0.765604 seconds


RateLimitError: Error code: 429 - {'error': {'message': 'Rate limit reached for gpt-3.5-turbo in organization org-nAjFWfh66NCMeot2zPIdLzug on requests per day (RPD): Limit 10000, Used 10000, Requested 1. Please try again in 8.64s. Visit https://platform.openai.com/account/rate-limits to learn more.', 'type': 'requests', 'param': None, 'code': 'rate_limit_exceeded'}}